<details>
<summary><b>Info</b></summary>

**Last Execution:** 2026-07-25

| Package | Version |
|---------|---------|
| **nnsight** | **0.8** |
| Python | 3.12.13 |
| torch | 2.13.0+cu126 |
| transformers | 5.15.0 |

</details>


# Module Access

nnsight wraps every model in an **Envoy** tree that mirrors the underlying
`torch.nn.Module` hierarchy. Every submodule is reachable by the same attribute
path you would use in plain PyTorch (`model.transformer.h[0].mlp`), and each one
exposes its live values during a forward pass through `.output`, `.input`, and
`.inputs`.

This page covers how to find your way around that tree: printing it, navigating
by attribute and index, giving modules portable aliases with `rename=`, calling a
module directly as a function, and reaching operations *inside* a forward with
`.source`.

## Setup

`TransformersModel` is the primary HuggingFace class in nnsight 0.8 (the older
`LanguageModel` still works but is a deprecated alias). It loads any model on the
Hub and, with `dispatch=True`, downloads the weights immediately.

In [1]:
from nnsight.modeling.transformers import TransformersModel
import torch

model = TransformersModel("openai-community/gpt2", device_map="auto", dispatch=True)

/home/localjadenfk/miniconda3/envs/ndif2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Printing the Envoy Tree

`print(model)` renders the whole module tree. This is your map: every name and
index shown here is a valid attribute path for accessing activations.

In [2]:
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
  (generator): Generator(
    (streamer): Streamer()
  )
)


## Navigating by Attribute and Index

Read any name off the printed tree to reach that Envoy. `ModuleList`s (like the
stack of transformer blocks) are indexed, and negative indices work as usual.

In [3]:
# A single block, its attention submodule, and its MLP.
block = model.transformer.h[0]
print(block.attn)
print(block.mlp)

GPT2Attention(
  (c_attn): Conv1D()
  (c_proj): Conv1D()
  (attn_dropout): Dropout(p=0.1, inplace=False)
  (resid_dropout): Dropout(p=0.1, inplace=False)
)
GPT2MLP(
  (c_fc): Conv1D()
  (c_proj): Conv1D()
  (act): NewGELUActivation()
  (dropout): Dropout(p=0.1, inplace=False)
)


Inside a trace, each Envoy exposes the values flowing through it via `.output`.
What that value *is* depends on the model architecture and your `transformers`
version — a transformer block might hand back a bare tensor, or a tuple whose
first element is the hidden states, or something else. Never assume; verify with
`print(module)`, `type(...)`, or `.shape`. For this GPT-2 build the last block's
`.output` comes back as a plain tensor of shape `(batch, seq, hidden)`, as the
cell below shows — so it can be indexed directly rather than via `output[0]`.

In [4]:
with model.trace("The Eiffel Tower is in the city of"):
    hidden = model.transformer.h[-1].output.save()

print(f"Last block output: {type(hidden).__name__}, shape {tuple(hidden.shape)}")

Last block output: Tensor, shape (1, 10, 768)


<details class="admonition note">
<summary>Reading vs. modifying</summary>

`.output` returns the real runtime tensor — there is no proxy to unwrap, so
`.shape`, `.mean()`, and `print` all work directly. Assigning `module.output = x`
replaces the value; `module.output[:] = 0` edits it in place. See the
*Getting Activations* and *Setting Activations* pages for details.

</details>

## Renaming Modules

Different architectures name the same role differently (`transformer.h` vs
`model.layers` vs `gpt_neox.layers`). Pass `rename={...}` at construction to
install aliases so your intervention code is portable across model families. An
alias points at the **same** Envoy object, so the original path keeps working too.

In [5]:
renamed = TransformersModel(
    "openai-community/gpt2",
    device_map="auto",
    dispatch=True,
    rename={
        "transformer.h": "layers",  # mount a subtree at a shorter path
        "mlp": "ffn",               # rename every block's MLP
    },
)

# Alias and original path resolve to the exact same Envoy.
print(renamed.layers[0].ffn is renamed.transformer.h[0].mlp)

with renamed.trace("Hello"):
    a = renamed.layers[0].ffn.output.save()       # via aliases
    b = renamed.transformer.h[0].mlp.output.save()  # original still works

print(torch.equal(a, b))

True
True


## Calling a Module as a Function

Inside a trace you can call any module as a function on values you already have.
This runs the module's `.forward()` directly — no hooks fire and execution order
is not enforced — which makes it perfect for applying a module *out of place*.

In [6]:
with model.trace("The Eiffel Tower is in the city of"):

    hs = model.transformer.h[-1].output

    # Apply the final layer norm then lm_head to decode the hidden states.
    logits = model.lm_head(model.transformer.ln_f(hs))
    token = logits[0, -1].argmax(dim=-1).save()

print(f"Decoded: {model.tokenizer.decode(token)}")

Decoded:  Paris


<details class="admonition note">
<summary>Function calls vs. attribute access</summary>

When you call a module (e.g. `model.lm_head(x)`), it runs `.forward()` directly:
no hooks fire and `.output` is unaffected. When you read `.output` or `.input`,
you get the values from the model's actual forward pass with full interleaving.
Pass `hook=True` to opt back into the full `module(...)` path (its hooks fire and
its submodules become observable).

</details>

## Logit Lens

A classic application: decode *every* layer's hidden states through the final
layer norm and lm_head to see what the model "thinks" at each layer.

In [7]:
with model.trace("The Eiffel Tower is in the city of"):

    predictions = list().save()

    for layer in model.transformer.h:
        hs = layer.output
        logits = model.lm_head(model.transformer.ln_f(hs))
        predictions.append(logits[0, -1].argmax(dim=-1))

for i, tok in enumerate(predictions):
    print(f"Layer {i:2d}: {model.tokenizer.decode(tok)}")

Layer  0:  the
Layer  1:  the
Layer  2:  the
Layer  3:  the
Layer  4:  the
Layer  5:  the
Layer  6:  East
Layer  7:  Ing
Layer  8:  Rome
Layer  9:  London
Layer 10:  Paris
Layer 11:  Paris


Notice how the prediction converges toward "Paris" in the later layers — this is
the logit lens in action. Each layer's intermediate representation is projected
into vocabulary space using the same final modules.

## Combining Module Calls with Torch Operations

Everything inside a trace is real PyTorch, so you can freely mix module calls with
standard torch operations.

In [8]:
with model.trace("The Eiffel Tower is in the city of"):

    hs = model.transformer.h[-1].output
    normed = model.transformer.ln_f(hs)

    # Cosine similarity between the last token and every earlier token.
    similarity = torch.cosine_similarity(
        normed[:, -1:, :], normed[:, :-1, :], dim=-1
    ).save()

print(f"Cosine similarity with last token: {similarity[0]}")

Cosine similarity with last token: tensor([0.9684, 0.9578, 0.9684, 0.9637, 0.9910, 0.9959, 0.9829, 0.9934, 0.9821],
       device='cuda:0', grad_fn=<SelectBackward0>)


## Reaching Inside a Forward with `.source`

`.output` and `.input` hook a module's boundaries. When the value you need lives
*between* two operations of a forward — and there is no submodule to attach to —
`module.source` exposes each call site in the module's `forward` as a hookable
operation. `print(module.source)` lists them (this works outside a trace):

In [9]:
print(model.transformer.h[0].mlp.source)

                    * def forward(self, hidden_states: tuple[torch.FloatTensor] | None) -> torch.FloatTensor:
 self_c_fc_0    ->  0     hidden_states = self.c_fc(hidden_states)
 self_act_0     ->  1     hidden_states = self.act(hidden_states)
 self_c_proj_0  ->  2     hidden_states = self.c_proj(hidden_states)
 self_dropout_0 ->  3     hidden_states = self.dropout(hidden_states)
                    4     return hidden_states
                    5 


Each labelled name (`self_act_0`, `self_c_proj_0`, ...) is an operation you can
read or replace inside a trace, just like a module's `.output`:

In [10]:
with model.trace("The Eiffel Tower is in the city of"):
    activation = model.transformer.h[0].mlp.source.self_act_0.output.save()

print(f"GELU activation shape: {tuple(activation.shape)}")

GELU activation shape: (1, 10, 3072)


See the *Accessing Intermediate Operations* page for the full `.source` story,
including recursive drilling into called functions.

## Gotchas

A couple of rules govern how you access the tree.

**Access modules in forward-pass order within an invoke.** Requesting a later
module and then an earlier one deadlocks and raises `OutOfOrderError`. To read
modules out of order, use separate invokes.

In [11]:
from nnsight.intervention.interleaver import OutOfOrderError

try:
    with model.trace("Hello world"):
        later = model.transformer.h[5].output.save()
        earlier = model.transformer.h[2].output.save()  # runs before h[5]
except OutOfOrderError as e:
    print(f"OutOfOrderError: {e}")

OutOfOrderError: 'model.transformer.h.2.output.i0' was requested but the model already ran past it


**`.save()` only makes sense inside a trace.** In nnsight 0.8 calling `.save()`
outside a tracing context raises (it used to be a silent no-op). Inside a trace it
marks a value to survive past the `with` block, and the value comes back under the
variable name you assigned it to.

In [12]:
try:
    torch.zeros(3).save()
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: save() was called outside a trace. `.save()` / nnsight.save(x) marks a value to return from the enclosing `with model.trace(...):` block, so it only works inside one — move the save into the trace block.
